# Default Deep Learning Models

In [38]:
""""
Deep learnig pipeline (default models) with:
- GPU-first attemp
- Walk-forward validation (expanding or fixed window)
- Models: LSTM, GRU, TCN, Casual (masked) Transformer (decoder-only)
- EarlyStopping + ModelCheckpoint
- Save: SavedModel, history.csv, preds/tunes .npy, meta.json, per-process metrics CSV, global CSV
- Default hyperparameters (tuning to ba added later)
"""

import os
import json
from pathlib import Path
import datetime
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [39]:
# Config

INPUT_FILES = [
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process1.csv",
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process2.csv",
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process3.csv"
]

OUT_DIR = Path(r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 25
HORIZON = 1
WINDOW_SIZE = None  # None = expanding window
STEP = 1
TEST_SIZE = 1

# default training params (per-model defaults)
DEFAULT_TRAINING = {
    "LSTM": {"epochs": 20, "batch_size": 32},
    "GRU": {"epochs": 20, "batch_size": 32},
    "TCN": {"epochs": 20, "batch_size": 32},
    "Transformer": {"epochs": 50, "batch_size": 32}
}

# general callbacks config
EARLY_PATIENCE = 7
VERBOSE = 1

# GPU performance (attempt to hid CPU devices; Tf may still do some CPU I/O)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_LOG_LEVEL"] = "3"

try:
    tf.config.set_visible_devices([], "CPU")
except Exception:
    # if it fails, proceed - TF sometimes refuses to remove CPU devices entirely.
    pass


In [40]:
# Utilities: metrics / file helpers
def timestamp():
    return datetime.datetime.now().strftime("%Y%m%dT%H%M%S")

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def SMAPE(y_true, y_pred):
    y_true = np.array(y_true).astype(float)
    y_pred = np.array(y_pred).astype(float)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-9)))

def directional_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if len(y_true) < 2:
        return float("nan")
    return float(np.mean((np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))).astype(int)))

def compute_metrics(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    da = directional_accuracy(y_true, y_pred)
    smape = SMAPE(y_true, y_pred)
    return {"MSE": mse, "MAE": mae, "RMSE": rmse, "DA": da, "SMAPE": smape}


In [41]:
# Data preparation - sequence from dataframe

def prepare_sequence_data(df, feature_cols, seq_len, horizon=1):
    X, y = [], []
    values = df[feature_cols].values.astype(np.float32)
    target = df["x"].values.astype(np.float32)
    for i in range(len(df) - seq_len - horizon + 1):
        X.append(values[i:i+seq_len])
        y.append(target[i + seq_len + horizon - 1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


In [42]:
def walk_forward_indices(n_samples, seq_len, horizon, window_size=None, step=1, test_size=1):
    """
    Generates (train_start, train_end, test_start, test_end) for walk-forward validation.
    
    train_end is exclusive.
    test_end is exclusive.
    """
    # minimum length of training window to even form one sequence
    min_train_len = seq_len + horizon  

    # initial train window
    train_start = 0
    train_end = min_train_len

    while True:
        test_start = train_end
        test_end = test_start + test_size   # Always defined here

        # STOP if test_end exceeds dataset size
        if test_end > n_samples:
            break

        # yield current fold
        yield (train_start, train_end, test_start, test_end)

        # Move the window forward
        train_end += step

        # Optional: fixed-size window
        if window_size is not None:
            train_start = max(0, train_end - window_size)


In [43]:
from tensorflow.keras.layers import Conv1D, Add, ReLU, GlobalAveragePooling1D, Input, Dense 

In [44]:
# model builders - default
def build_lstm(input_shape, units=64, dense_units=32):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(units, return_sequences=False, kernel_initializer="he_normal"),
        tf.keras.layers.Dense(dense_units, activation="relu"),
        tf.keras.layers.Dense(1)  
    ])
    model.compile(optimizer=tf.keras.optimizer.Adam(), loss="mse")
    return model

def build_gru(input_shape, units=64, dense_units=32):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.GRU(units, return_sequences=False, kernel_initializer="he_normal"),
        tf.keras.layers.Dense(dense_units, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizer.Adam(), loss="mse")
    return model

# TCN - simple casual conv residual blocks
def build_tcn(input_shape, filters=32, kernel_size=3, dilations=(1,2,4,8), dense_units=32):
    inp = Input(shape=input_shape)
    x = inp
    for d in dilations:
        prev = x
        x = Conv1D(filters, kernel_size, padding="casual", dilation_rate=d, activation="relu")(x)
        x = Conv1D(filters, kernel_size, padding="casual", dilation_rate=d, activation="relu")(x)
        # residuals (if channels differ, project)
        if prev.shape[-1] != x.shape[1]:
            prev = tf.keras.layers.Dense(filters)(prev)
        x = Add()([prev, x])
        x = ReLU()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(dense_units, activation="relu")(x)
    out = Dense(1)(x)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss="mse")
    return model

# Casual Masked Transformer
def create_casual_mask(batch_size, seq_len):
    # lower triangular ones => True where allowed
    tril = tf.linlag.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0)
    tril = tf.expand_dim(tril, 0)
    return tf.tile(tril, [batch_size, 1, 1])  # (batch, seq_len, seq_len)

def transformer_block(x, seq_len, embed_dim, num_heads, ff_dim, dropout_rate, training):
    ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    # build masked MHA inside a lambda to construct mask with runtime batch size
    def mha_call(q):
        batch = tf.shape(1)[0]
        mask = create_casual_mask(batch, seq_len) # boolean mask
        attn_out = tf.keras.layers.MultiHeadAttention(
            num_heads = num_heads, key_dim=embed_dim // num_heads, dropout=dropout_rate
        )(q, q, attention_mask=mask, training=training)
        return attn_out
    
    attn_out = tf.keras.layers.Lambda(mha_call)(ln1)
    attn_out = tf.keras.layers.dropout(dropout_rate)(attn_out, training=training)
    x = tf.keras.layers.Add()([x, attn_out])

    ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    ff = tf.keras.layers.Dense(ff_dim, activation='relu')(ln2)
    ff = tf.keras.layers.Dense(embed_dim)(ff)
    ff = tf.keras.layers.Dropout(dropout_rate)(ff, training=training)
    x = tf.keras.layers.Add()([x, ff])
    return x

def build_transformer(input_shape,
                      seq_len=None,
                      embed_dim=64,
                      num_heads=4,
                      ff_dim=256,
                      num_layers=3,
                      dropout_rate=0.1,
                      final_activation=None):
    assert len(input_shape) == 2, "input must be (seq_len, n_features)"
    if seq_len is None:
        seq_len = input_shape[0]

        inputs = tf.keras.Input(shape=input_shape) # (batch, seq_len, n_features)
        # token projection
        x = tf.keras.layers.Dense(embed_dim)(inputs)

        # learned positional embeddings
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_embedding_layer = tf.keras.layers.Embedding(input_dim=seq_len, output_dim=embed_dim)
        pos_embeddings = pos_embedding_layer(positions) # (seq_len, embed_dim)
        pos_embeddings = tf.expand_dims(pos_embeddings, 0)
        x = x + pos_embeddings

        x = tf.keras.layers.Dropout(dropout_rate)(x)

        for _ in range(num_layers):
            # use training=False default during build; during fit Keras will set training appropriately
            x = transformer_block(x, seq_len, embed_dim, num_heads, ff_dim, dropout_rate, training=False)
        
        x = tf.keras.layers.Layers.LayerNomalization(epsilon=1e-6)(x)
        last_token = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)
        out = tf.keras.layers.Dense(64, activation='relu')(last_token)
        out = tf.keras.layers.Dropout(dropout_rate)(out)
        out = tf.keras.layers.Dense(1, activation=final_activation)(out)

        model = tf.keras.Model(inputs=inputs, outputs=out)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss = 'mse')
        return model
    
    # mapping
MODEL_BUILDERS = {
        "LSTM": build_lstm,
        "GRU": build_gru,
        "TCN": build_tcn,
        "Transformer": build_transformer
    }


In [45]:
# train + save loop (walk-forward)

def save_training_history(history, path: Path):
    # history is tf.keras.callbacks.History.history dict
    pd.DataFrame(history).to_csv(path, index=False)

def save_meta(path: Path, meta: dict):
    with open(path, "w") as fh:
        json.dump(meta, fh, indent=2)

def run_process(input_csv_path: str,
                out_dir: Path,
                seq_len=SEQ_LEN,
                horizon=HORIZON,
                window_size=WINDOW_SIZE,
                step=STEP,
                test_size=TEST_SIZE,
                model_builders=MODEL_BUILDERS,
                default_training=DEFAULT_TRAINING):
    df = pd.read_csv(input_csv_path)
    process_name = Path(input_csv_path).stem
    feature_cols = [c for c in df.columns if c != "x"]

    # create directories
    proc_out = out_dir / process_name
    ensure_dir(proc_out)
    models_out = proc_out / "models"
    ensure_dir(models_out)
    preds_out = proc_out / "predictions"
    ensure_dir(preds_out)
    joblib_out = proc_out / "joblib"
    ensure_dir(joblib_out)
    logs_out = proc_out / "logs"
    ensure_dir(logs_out)

    # per-process metrics csv
    summary_csv = proc_out / f"{process_name}_metrics.csv"
    if not summary_csv.exists():
        pd.DataFrame(columns=[
            "timestamp", "process", "model", "fold",
            "train_start_idx", "train_end_idx_excl", "test_start_idx", "test_end_idx_excl",
            "MSE", "MAE", "RMSE", "DA", "SMAPE",
            "model_path", "preds_path", "trues_path", "meta_path"
        ]).to_csv(summary_csv, index=False)

    # build X,y
    X, y = prepare_sequence_data(df, feature_cols, seq_len, horizon)
    n_samples = len(df)

    for model_name, builder in model_builders.items():
        model_dir = models_out / model_name
        ensure_dir(model_dir)
        fold = 0

        for (train_s, train_e, test_s, test_e) in walk_forward_indices(
                n_samples=n_samples, seq_len=seq_len, horizon=horizon, window_size=window_size, step=step, test_size=test_size):

            fold += 1
            # expanding window: train slice may be (0, start) and could be empty early
            if window_size is None:
                t0, t1 = 0, train_e
            else:
                t0, t1 = train_s, train_e

            X_train = X[t0:t1]
            y_train = y[t0:t1]
            X_test = X[test_s:test_e]
            y_test = y[test_s:test_e]

            # skip empty train
            if len(X_train) == 0:
                continue

            defaults = default_training.get(model_name, {"epochs": 20, "batch_size": 32})
            epochs = defaults["epochs"]
            batch_size = defaults["batch_size"]

            run_ts = timestamp()
            fold_id = f"{model_name}_fold{fold}_{run_ts}"
            # paths
            fold_model_dir = model_dir / fold_id
            ensure_dir(fold_model_dir)
            preds_path = preds_out / f"{process_name}_{fold_id}_preds.npy"
            trues_path = preds_out / f"{process_name}_{fold_id}_trues.npy"
            history_path = logs_out / f"{process_name}_{fold_id}_history.csv"
            meta_path = logs_out / f"{process_name}_{fold_id}_meta.json"

            # build model
            model = builder(input_shape=X.shape[1:])

            # callbacks: earlystop + checkpoint
            checkpoint_path = str(fold_model_dir / "best_ckpt.h5")
            cb_checkpoint = tf.keras.callbacks.ModelCheckpoint(
                checkpoint_path, monitor="val_loss", save_best_only=True, save_weights_only=False, verbose=0
            )
            cb_early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLYSTOP_PATIENCE, restore_best_weights=True, verbose=0)

            # small validation split from train for early stopping (10%)
            val_split = 0.1 if len(X_train) >= 10 else 0.2

            history = model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=batch_size,
                validation_split=val_split,
                callbacks=[cb_checkpoint, cb_early],
                verbose=VERBOSE
            )

            # after training, ensure best model is loaded (ModelCheckpoint saved best model)
            try:
                # load from checkpoint path if checkpoint saved a full model
                best_model = tf.keras.models.load_model(checkpoint_path)
            except Exception:
                # fallback: use model (weights may be already best due to restore_best_weights)
                best_model = model

            # predict on test
            y_pred = best_model.predict(X_test, verbose=0).flatten()
            y_true = y_test.flatten()

            # metrics
            metrics = compute_metrics(y_true, y_pred)

            # save model (SavedModel)
            saved_model_path = fold_model_dir / "saved_model"
            try:
                best_model.save(str(saved_model_path), include_optimizer=True)
            except Exception:
                # fallback to saving weights only
                best_model.save_weights(str(fold_model_dir / "weights.h5"))

            # save preds/trues/history/meta
            np.save(preds_path, y_pred)
            np.save(trues_path, y_true)
            save_training_history(history.history, history_path)

            meta = {
                "timestamp": run_ts,
                "process": process_name,
                "model": model_name,
                "fold": fold,
                "train_start_idx": int(t0),
                "train_end_idx_excl": int(t1),
                "test_start_idx": int(test_s),
                "test_end_idx_excl": int(test_e),
                "hyperparameters": {
                    "model_defaults": defaults,
                    # also store architecture-relevant settings if desired:
                    # For transformer we store defaults; for other models we store typical params
                    "model_specific_defaults": {
                        "Transformer": {"embed_dim": 64, "num_heads": 4, "ff_dim": 256, "num_layers": 3},
                        "LSTM": {"units": 64, "dense_units": 32},
                        "GRU": {"units": 64, "dense_units": 32},
                        "TCN": {"filters": 32, "kernel_size": 3, "dilations": [1,2,4,8]}
                    }.get(model_name, {})
                },
                "metrics": metrics,
                "paths": {
                    "saved_model": str(saved_model_path),
                    "preds": str(preds_path),
                    "trues": str(trues_path),
                    "history": str(history_path)
                }
            }
            save_meta(meta_path, meta)

            # append per-process summary CSV
            row = {
                "timestamp": run_ts, "process": process_name, "model": model_name, "fold": fold,
                "train_start_idx": int(t0), "train_end_idx_excl": int(t1),
                "test_start_idx": int(test_s), "test_end_idx_excl": int(test_e),
                "MSE": metrics["MSE"], "MAE": metrics["MAE"], "RMSE": metrics["RMSE"],
                "DA": metrics["DA"], "SMAPE": metrics["SMAPE"],
                "model_path": str(saved_model_path), "preds_path": str(preds_path),
                "trues_path": str(trues_path), "meta_path": str(meta_path)
            }
            pd.DataFrame([row]).to_csv(summary_csv, mode="a", header=False, index=False)

            # append to global summary
            global_summary = out_dir / "all_processes_summary.csv"
            if not global_summary.exists():
                pd.DataFrame([row]).to_csv(global_summary, index=False)
            else:
                pd.DataFrame([row]).to_csv(global_summary, mode="a", header=False, index=False)

            # cleanup
            del model, best_model
            tf.keras.backend.clear_session()

            print(f"[{process_name}] {model_name} fold{fold} done. MSE={metrics['MSE']:.6g} SMAPE={metrics['SMAPE']:.6g}")

    print(f"Finished process: {process_name} -> outputs in: {proc_out}")

# -----------------------------
# ENTRYPOINT
# -----------------------------
def main():
    for path in INPUT_FILES:
        run_process(path, OUT_DIR)

if __name__ == "__main__":
    main()

NotFoundError: No CPU devices are available in this process

In [60]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices())


2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [47]:
!pip uninstall -y tensorflow tensorflow-intel tensorflow-cpu


Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0


You can safely remove it manually.


In [49]:
!pip install tensorflow[and-cuda]


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 8.37.0 requires typing_extensions>=4.6; python_version < "3.12", but you have typing-extensions 4.5.0 which is incompatible.
exceptiongroup 1.3.1 requires typing-extensions>=4.6.0; python_version < "3.13", but you have typing-extensions 4.5.0 which is incompatible.
fastapi 0.123.1 requires typing-extensions>=4.8.0, but you have typing-extensions 4.5.0 which is incompatible.
optree 0.18.0 requires typing-extensions>=4.6.0, but you have typing-extensions 4.5.0 which is incompatible.
pydantic 2.12.5 requires typing-extensions>=4.14.1, but you have typing-extensions 4.5.0 which is incompatible.
pydantic-core 2.41.5 requires typing-extensions>=4.14.1, but you have typing-extensions 4.5.0 which is incom

  Using cached tensorflow-2.20.0-cp310-cp310-win_amd64.whl.metadata (4.6 kB)
INFO: pip is looking at multiple versions of tensorflow[and-cuda] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tensorflow[and-cuda] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of grpcio to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/276.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/276.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/276.5 MB ? eta -:--:--
   --

In [55]:
import tensorflow as tf
print(tf.config.list_physical_devices())


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [57]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[]


In [59]:
!nvidia-smi


Wed Dec 10 01:46:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.97                 Driver Version: 555.97         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   55C    P3             28W /   51W |       0MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices("GPU"))


GPU Available: []


In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

from nbeats_keras.model import NBeatsNet
print("NBeats loaded successfully")


2.20.0
[]
NBeats loaded successfully


In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))


ModuleNotFoundError: No module named 'tensorflow'

In [2]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))


2.11.1
[]


In [58]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
    tf.config.experimental.set_memory_growth(gpus[0], True)
print("Using:", tf.config.list_logical_devices())


NotFoundError: No CPU devices are available in this process

In [61]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices())


2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
